# Cat?logo inicial de queries y descarga de CSV con yfinance para el TFM

Este notebook deja las salidas directamente junto a su ubicaci?n:

- `query_cases.json`: cat?logo de queries
- `data/`: un CSV por query descargado desde `yfinance`
- `data/*.metadata.json`: metadatos m?nimos por descarga


## 1. Librer?as y rutas de trabajo

Este notebook trabaja directamente junto a su ubicaci?n con dos salidas simples:

- `query_cases.json`: cat?logo de queries
- `data/`: carpeta donde se guardan los CSV hist?ricos y sus metadatos


In [ ]:
from __future__ import annotations

import json
import re
import unicodedata
from datetime import date, datetime
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
QUERY_CASES_JSON_PATH = NOTEBOOK_DIR / "query_cases.json"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TODAY = date.today()

NOTEBOOK_DIR, DATA_DIR, QUERY_CASES_JSON_PATH

(WindowsPath('c:/Users/usuario/Desktop/tfm/notebooks'),
 WindowsPath('c:/Users/usuario/Desktop/tfm/notebooks/data'),
 WindowsPath('c:/Users/usuario/Desktop/tfm/notebooks/query_cases.json'))

## 2. Catálogo curado de queries iniciales

Aquí fijamos un conjunto pequeño pero representativo de casos de uso para arrancar el TFM.  
Cada fila incluye la `query`, la `analysis_goal`, los `tickers`, el rango temporal y campos de control como `needs_clarification` y `warnings`.


In [14]:
QUERY_CASES = [
    {
        "query": "Cuánto ha crecido Nvidia en 5 años",
        "analysis_goal": "price_growth",
        "tickers": ["NVDA"],
        "start": None,
        "end": None,
        "period": "5y",
        "interval": "1d",
        "needs_clarification": False,
        "warnings": "",
    },
    {
        "query": "Descárgame el histórico del S&P 500 desde 2020",
        "analysis_goal": "historical_download",
        "tickers": ["^GSPC"],
        "start": "2020-01-01",
        "end": None,
        "period": None,
        "interval": "1d",
        "needs_clarification": False,
        "warnings": "",
    },
    {
        "query": "Quiero el oro en 1 semana a 1h",
        "analysis_goal": "historical_download",
        "tickers": ["GC=F"],
        "start": None,
        "end": None,
        "period": "1wk",
        "interval": "1h",
        "needs_clarification": False,
        "warnings": "",
    },
    {
        "query": "Compara Nvidia y AMD en 2 años",
        "analysis_goal": "compare_assets",
        "tickers": ["NVDA", "AMD"],
        "start": None,
        "end": None,
        "period": "2y",
        "interval": "1d",
        "needs_clarification": False,
        "warnings": "",
    },
    {
        "query": "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
        "analysis_goal": "historical_download",
        "tickers": ["BTC-USD"],
        "start": "2024-01-01",
        "end": "2024-12-31",
        "period": None,
        "interval": "1d",
        "needs_clarification": False,
        "warnings": "",
    },
    {
        "query": "QQQ y SPY desde 2024-01-01 hasta 2024-12-31",
        "analysis_goal": "historical_download",
        "tickers": ["QQQ", "SPY"],
        "start": "2024-01-01",
        "end": "2024-12-31",
        "period": None,
        "interval": "1d",
        "needs_clarification": False,
        "warnings": "",
    },
    {
        "query": "AAPL en 3 meses",
        "analysis_goal": "historical_download",
        "tickers": ["AAPL"],
        "start": None,
        "end": None,
        "period": "3mo",
        "interval": "1d",
        "needs_clarification": False,
        "warnings": "",
    },
    {
        "query": "EUR/USD en 10 días a 1h",
        "analysis_goal": "historical_download",
        "tickers": ["EURUSD=X"],
        "start": None,
        "end": None,
        "period": "10d",
        "interval": "1h",
        "needs_clarification": False,
        "warnings": "",
    },
]

queries_df = pd.DataFrame(QUERY_CASES)
queries_df


,query,analysis_goal,tickers,start,end,period,interval,needs_clarification,warnings
0,Cuánto ha crecido Nvidia en 5 años,price_growth,[NVDA],None,None,5y,1d,False,
1,Descárgame el histórico del S&P 500 desde 2020,historical_download,[^GSPC],2020-01-01,None,None,1d,False,
2,Quiero el oro en 1 semana a 1h,historical_download,[GC=F],None,None,1wk,1h,False,
3,Compara Nvidia y AMD en 2 años,compare_assets,"[NVDA, AMD]",None,None,2y,1d,False,
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,historical_download,[BTC-USD],2024-01-01,2024-12-31,None,1d,False,
5,QQQ y SPY desde 2024-01-01 hasta 2024-12-31,historical_download,"[QQQ, SPY]",2024-01-01,2024-12-31,None,1d,False,
6,AAPL en 3 meses,historical_download,[AAPL],None,None,3mo,1d,False,
7,EUR/USD en 10 días a 1h,historical_download,[EURUSD=X],None,None,10d,1h,False,


## 3. Exportación del catálogo a JSON

Aquí guardamos exactamente `QUERY_CASES` en un JSON reutilizable junto al notebook.  
Las descargas de mercado quedan separadas dentro de la carpeta `data/`.


In [15]:
QUERY_CASES_JSON_PATH.write_text(
    json.dumps(QUERY_CASES, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

QUERY_CASES_JSON_PATH


WindowsPath('c:/Users/usuario/Desktop/tfm/notebooks/query_cases.json')

In [16]:
json.loads(QUERY_CASES_JSON_PATH.read_text(encoding="utf-8"))


[{'query': 'Cuánto ha crecido Nvidia en 5 años',
  'analysis_goal': 'price_growth',
  'tickers': ['NVDA'],
  'start': None,
  'end': None,
  'period': '5y',
  'interval': '1d',
  'needs_clarification': False,
  'warnings': ''},
 {'query': 'Descárgame el histórico del S&P 500 desde 2020',
  'analysis_goal': 'historical_download',
  'tickers': ['^GSPC'],
  'start': '2020-01-01',
  'end': None,
  'period': None,
  'interval': '1d',
  'needs_clarification': False,
  'warnings': ''},
 {'query': 'Quiero el oro en 1 semana a 1h',
  'analysis_goal': 'historical_download',
  'tickers': ['GC=F'],
  'start': None,
  'end': None,
  'period': '1wk',
  'interval': '1h',
  'needs_clarification': False,
  'warnings': ''},
 {'query': 'Compara Nvidia y AMD en 2 años',
  'analysis_goal': 'compare_assets',
  'tickers': ['NVDA', 'AMD'],
  'start': None,
  'end': None,
  'period': '2y',
  'interval': '1d',
  'needs_clarification': False,
  'warnings': ''},
 {'query': 'Datos de Bitcoin desde 2024-01-01 hasta

## 4. Funciones auxiliares de validación y normalización

Estas reglas son deliberadamente simples, pero ayudan a dejar limpio el dataset de partida:

- comprobar que haya al menos un ticker;
- evitar que se mezclen `period` y `start/end` sin control;
- marcar incompatibilidades entre rango e intervalo intradía;
- comprobar que `compare_assets` tenga al menos dos activos;
- generar nombres de fichero seguros para cada consulta.


In [17]:
VALID_ANALYSIS_GOALS = {
    "asset_overview",
    "price_growth",
    "compare_assets",
    "return_analysis",
    "historical_risk_analysis",
    "technical_analysis",
    "historical_download",
}

INTRADAY_INTERVALS = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h"}
PERIOD_TO_DAYS = {"d": 1, "wk": 7, "mo": 30, "y": 365}

def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = text.replace(" ", "_")
    text = re.sub(r"[^a-z0-9_]+", "", text)
    return text[:100].strip("_") or "query"

def estimate_days_from_period(period: str | None) -> int | None:
    if not period:
        return None
    match = re.fullmatch(r"(\d+)(d|wk|mo|y)", period)
    if not match:
        return None
    n, unit = match.groups()
    return int(n) * PERIOD_TO_DAYS[unit]

def estimate_days_from_dates(start: str | None, end: str | None) -> int | None:
    try:
        if start and end:
            start_dt = datetime.fromisoformat(start).date()
            end_dt = datetime.fromisoformat(end).date()
            return (end_dt - start_dt).days
        if start and not end:
            start_dt = datetime.fromisoformat(start).date()
            return (TODAY - start_dt).days
    except Exception:
        return None
    return None

def validate_row(row: pd.Series) -> pd.Series:
    warnings = []
    tickers = row["tickers"]
    analysis_goal = row["analysis_goal"]
    period = row["period"]
    start = row["start"]
    end = row["end"]
    interval = row["interval"]
    needs_clarification = bool(row["needs_clarification"])

    if analysis_goal not in VALID_ANALYSIS_GOALS:
        warnings.append(f"Analysis goal no contemplada en el alcance actual: {analysis_goal}")

    if not isinstance(tickers, list) or len(tickers) == 0:
        warnings.append("No hay tickers definidos.")
        needs_clarification = True

    if analysis_goal == "compare_assets" and isinstance(tickers, list) and len(tickers) < 2:
        warnings.append("compare_assets requiere al menos 2 tickers.")
        needs_clarification = True

    if period and (start or end):
        warnings.append("Hay period y start/end a la vez; conviene priorizar uno solo.")

    if not period and not start:
        warnings.append("Falta rango temporal: define period o start.")
        needs_clarification = True

    span_days = estimate_days_from_period(period)
    if span_days is None:
        span_days = estimate_days_from_dates(start, end)

    if interval in INTRADAY_INTERVALS and span_days is not None and span_days > 60:
        warnings.append(
            f"Intervalo intradía ({interval}) con ventana estimada > 60 días ({span_days}). "
            "Yahoo Finance suele limitar este tipo de descarga."
        )

    row["warnings"] = " | ".join(warnings)
    row["needs_clarification"] = needs_clarification
    return row

validated_df = queries_df.copy().apply(validate_row, axis=1)
validated_df


,query,analysis_goal,tickers,start,end,period,interval,needs_clarification,warnings
0,Cuánto ha crecido Nvidia en 5 años,price_growth,[NVDA],None,None,5y,1d,False,
1,Descárgame el histórico del S&P 500 desde 2020,historical_download,[^GSPC],2020-01-01,None,None,1d,False,
2,Quiero el oro en 1 semana a 1h,historical_download,[GC=F],None,None,1wk,1h,False,
3,Compara Nvidia y AMD en 2 años,compare_assets,"[NVDA, AMD]",None,None,2y,1d,False,
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,historical_download,[BTC-USD],2024-01-01,2024-12-31,None,1d,False,
5,QQQ y SPY desde 2024-01-01 hasta 2024-12-31,historical_download,"[QQQ, SPY]",2024-01-01,2024-12-31,None,1d,False,
6,AAPL en 3 meses,historical_download,[AAPL],None,None,3mo,1d,False,
7,EUR/USD en 10 días a 1h,historical_download,[EURUSD=X],None,None,10d,1h,False,


In [18]:
validated_df


,query,analysis_goal,tickers,start,end,period,interval,needs_clarification,warnings
0,Cuánto ha crecido Nvidia en 5 años,price_growth,[NVDA],None,None,5y,1d,False,
1,Descárgame el histórico del S&P 500 desde 2020,historical_download,[^GSPC],2020-01-01,None,None,1d,False,
2,Quiero el oro en 1 semana a 1h,historical_download,[GC=F],None,None,1wk,1h,False,
3,Compara Nvidia y AMD en 2 años,compare_assets,"[NVDA, AMD]",None,None,2y,1d,False,
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,historical_download,[BTC-USD],2024-01-01,2024-12-31,None,1d,False,
5,QQQ y SPY desde 2024-01-01 hasta 2024-12-31,historical_download,"[QQQ, SPY]",2024-01-01,2024-12-31,None,1d,False,
6,AAPL en 3 meses,historical_download,[AAPL],None,None,3mo,1d,False,
7,EUR/USD en 10 días a 1h,historical_download,[EURUSD=X],None,None,10d,1h,False,


## 5. Vista previa de ficheros a descargar

Este paso solo muestra qué CSV y metadatos se generarán dentro de `data/`.


In [19]:
manifest_df = validated_df.copy()
manifest_df["csv_filename"] = manifest_df["query"].apply(lambda q: f"{slugify(q)}.csv")
manifest_df["metadata_filename"] = manifest_df["query"].apply(lambda q: f"{slugify(q)}.metadata.json")

manifest_df[["query", "csv_filename", "metadata_filename", "warnings"]]


,query,csv_filename,metadata_filename,warnings
0,Cuánto ha crecido Nvidia en 5 años,cuanto_ha_crecido_nvidia_en_5_anos.csv,cuanto_ha_crecido_nvidia_en_5_anos.metadata.json,
1,Descárgame el histórico del S&P 500 desde 2020,descargame_el_historico_del_sp_500_desde_2020.csv,descargame_el_historico_del_sp_500_desde_2020....,
2,Quiero el oro en 1 semana a 1h,quiero_el_oro_en_1_semana_a_1h.csv,quiero_el_oro_en_1_semana_a_1h.metadata.json,
3,Compara Nvidia y AMD en 2 años,compara_nvidia_y_amd_en_2_anos.csv,compara_nvidia_y_amd_en_2_anos.metadata.json,
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,datos_de_bitcoin_desde_20240101_hasta_20241231...,datos_de_bitcoin_desde_20240101_hasta_20241231...,
5,QQQ y SPY desde 2024-01-01 hasta 2024-12-31,qqq_y_spy_desde_20240101_hasta_20241231.csv,qqq_y_spy_desde_20240101_hasta_20241231.metada...,
6,AAPL en 3 meses,aapl_en_3_meses.csv,aapl_en_3_meses.metadata.json,
7,EUR/USD en 10 días a 1h,eurusd_en_10_dias_a_1h.csv,eurusd_en_10_dias_a_1h.metadata.json,


## 6. Preparación de parámetros para `yfinance`

Esta función transforma una fila del catálogo en el diccionario que se puede pasar a `yf.download(...)`.


In [20]:
def row_to_yfinance_kwargs(row: pd.Series) -> dict:
    params = {
        "tickers": row["tickers"],
        "interval": row["interval"],
        "group_by": "ticker",
        "auto_adjust": False,
        "threads": True,
        "progress": False,
    }

    if row["period"]:
        params["period"] = row["period"]
    else:
        params["start"] = row["start"]
        if row["end"]:
            params["end"] = row["end"]
    return params

example_params = row_to_yfinance_kwargs(validated_df.iloc[0])
example_params


{'tickers': ['NVDA'],
 'interval': '1d',
 'group_by': 'ticker',
 'auto_adjust': False,
 'threads': True,
 'progress': False,
 'period': '5y'}

## 7. Descarga y guardado de un CSV por consulta

Por defecto dejamos `DO_REAL_DOWNLOAD = False` para que el notebook no falle en entornos sin red.  
Cuando lo ejecutes en un entorno con acceso a Yahoo Finance, bastará con ponerlo a `True`.


In [21]:
# Si hace falta, descomenta:
# !pip install -q yfinance

DO_REAL_DOWNLOAD = False


In [22]:
def save_metadata(row: pd.Series, output_dir: Path) -> Path:
    metadata = {
        "query": row["query"],
        "analysis_goal": row["analysis_goal"],
        "tickers": row["tickers"],
        "start": row["start"],
        "end": row["end"],
        "period": row["period"],
        "interval": row["interval"],
        "needs_clarification": bool(row["needs_clarification"]),
        "warnings": row["warnings"],
    }
    metadata_path = output_dir / f"{slugify(row['query'])}.metadata.json"
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    return metadata_path

def download_case(row: pd.Series, output_dir: Path = DATA_DIR) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"{slugify(row['query'])}.csv"
    metadata_path = save_metadata(row, output_dir)

    if bool(row["needs_clarification"]):
        return {
            "query": row["query"],
            "status": "skipped_needs_clarification",
            "csv_path": None,
            "metadata_path": str(metadata_path),
            "rows": 0,
        }

    try:
        import yfinance as yf
        df = yf.download(**row_to_yfinance_kwargs(row))

        if df.empty:
            return {
                "query": row["query"],
                "status": "empty_or_network_error",
                "csv_path": None,
                "metadata_path": str(metadata_path),
                "rows": 0,
            }

        df.to_csv(csv_path)
        return {
            "query": row["query"],
            "status": "ok",
            "csv_path": str(csv_path),
            "metadata_path": str(metadata_path),
            "rows": len(df),
        }
    except Exception as exc:
        return {
            "query": row["query"],
            "status": f"error: {type(exc).__name__}",
            "csv_path": None,
            "metadata_path": str(metadata_path),
            "rows": 0,
            "detail": str(exc),
        }


In [23]:
if DO_REAL_DOWNLOAD:
    results = [download_case(row) for _, row in validated_df.iterrows()]
    download_results_df = pd.DataFrame(results)
else:
    results = [
        {
            "query": row["query"],
            "status": "dry_run",
            "csv_path": str(DATA_DIR / f"{slugify(row['query'])}.csv"),
            "metadata_path": str(DATA_DIR / f"{slugify(row['query'])}.metadata.json"),
            "rows": None,
        }
        for _, row in validated_df.iterrows()
    ]
    download_results_df = pd.DataFrame(results)

download_results_df


,query,status,csv_path,metadata_path,rows
0,Cuánto ha crecido Nvidia en 5 años,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\cu...,c:\Users\usuario\Desktop\tfm\notebooks\data\cu...,None
1,Descárgame el histórico del S&P 500 desde 2020,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\de...,c:\Users\usuario\Desktop\tfm\notebooks\data\de...,None
2,Quiero el oro en 1 semana a 1h,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\qu...,c:\Users\usuario\Desktop\tfm\notebooks\data\qu...,None
3,Compara Nvidia y AMD en 2 años,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\co...,c:\Users\usuario\Desktop\tfm\notebooks\data\co...,None
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\da...,c:\Users\usuario\Desktop\tfm\notebooks\data\da...,None
5,QQQ y SPY desde 2024-01-01 hasta 2024-12-31,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\qq...,c:\Users\usuario\Desktop\tfm\notebooks\data\qq...,None
6,AAPL en 3 meses,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\aa...,c:\Users\usuario\Desktop\tfm\notebooks\data\aa...,None
7,EUR/USD en 10 días a 1h,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\eu...,c:\Users\usuario\Desktop\tfm\notebooks\data\eu...,None


In [24]:
download_results_df


,query,status,csv_path,metadata_path,rows
0,Cuánto ha crecido Nvidia en 5 años,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\cu...,c:\Users\usuario\Desktop\tfm\notebooks\data\cu...,None
1,Descárgame el histórico del S&P 500 desde 2020,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\de...,c:\Users\usuario\Desktop\tfm\notebooks\data\de...,None
2,Quiero el oro en 1 semana a 1h,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\qu...,c:\Users\usuario\Desktop\tfm\notebooks\data\qu...,None
3,Compara Nvidia y AMD en 2 años,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\co...,c:\Users\usuario\Desktop\tfm\notebooks\data\co...,None
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-1...,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\da...,c:\Users\usuario\Desktop\tfm\notebooks\data\da...,None
5,QQQ y SPY desde 2024-01-01 hasta 2024-12-31,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\qq...,c:\Users\usuario\Desktop\tfm\notebooks\data\qq...,None
6,AAPL en 3 meses,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\aa...,c:\Users\usuario\Desktop\tfm\notebooks\data\aa...,None
7,EUR/USD en 10 días a 1h,dry_run,c:\Users\usuario\Desktop\tfm\notebooks\data\eu...,c:\Users\usuario\Desktop\tfm\notebooks\data\eu...,None


## 8. Resultado

Este notebook deja dos salidas simples junto a su ubicación:

- `query_cases.json` con el catálogo de queries.
- `data/` con un CSV por query y su metadata asociada.
